In [6]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler

# 1. Load the dataset - NO df.sample() needed as undersampling was performed
df = pd.read_csv('Rider-Info.csv')
df = df.dropna(subset=['cancelled'])
df['cancelled'] = df['cancelled'].astype(int)

# 2. Preprocessing & Feature Engineering
df['allot_delay_min'] = (pd.to_datetime(df['allot_time'], errors='coerce') - pd.to_datetime(df['order_time'], errors='coerce')).dt.total_seconds() / 60

feature_cols = [
    'first_mile_distance', 'last_mile_distance', 
    'alloted_orders', 'delivered_orders', 'session_time',
    'allot_delay_min'
]

X = df[feature_cols].copy()
y = df['cancelled']

for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

# 3. Train-test split FIRST (keep test set completely untouched and imbalanced)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Apply Random Undersampling ONLY to the training set
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

# 5. Train Random Forest on the balanced training data
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_resampled, y_train_resampled)

# 6. Save with compression=3 so the file stays small for GitHub (<25MB)
joblib.dump(rf_model, 'random_forest_model.pkl', compress=3)
print("Model trained on full data, undersampled, and saved successfully!")

Model trained and saved successfully as 'random_forest_model.pkl'!
